<a href="https://colab.research.google.com/github/Amritpal986/-Amritpal986.github.io/blob/main/Copy_of_AI_Powered_Employee_Attrition_Prediction_%26_Workforce_Analytics_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

### Loading a sample dataset

Since `employee_data.xlsx` was not found, I'll load the 'titanic' dataset from Seaborn as a sample for demonstration purposes.

In [ ]:
import seaborn as sns

df = sns.load_dataset('titanic')
display(df.head())

In [ ]:
print(df.shape)
print(df.info())
print(df.describe())

In [ ]:
print(df.isnull().sum())

df.fillna(df.median(numeric_only=True), inplace=True)

In [ ]:
encoder = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = encoder.fit_transform(df[col])

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.show()

In [ ]:
X = df.drop('survived', axis=1)
y = df['survived']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

# Preprocessing X_train and X_test to handle remaining non-numeric data
# Drop 'deck' column due to high number of missing values
if 'deck' in X_train.columns:
    X_train = X_train.drop('deck', axis=1)
if 'deck' in X_test.columns:
    X_test = X_test.drop('deck', axis=1)

# Initialize LabelEncoder
encoder = LabelEncoder()

# Encode 'class' column which is of 'category' dtype and contains strings.
# Fit encoder on X_train 'class' and transform both X_train and X_test
if 'class' in X_train.columns:
    X_train['class'] = encoder.fit_transform(X_train['class'])
    # Use the same fitted encoder for X_test to ensure consistency
    X_test['class'] = encoder.transform(X_test['class'])

# Convert boolean columns to integer (0 or 1)
for col in X_train.select_dtypes(include='bool').columns:
    X_train[col] = X_train[col].astype(int)
for col in X_test.select_dtypes(include='bool').columns:
    X_test[col] = X_test[col].astype(int)


model.fit(X_train, y_train)

In [ ]:
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

In [ ]:
print(classification_report(
    y_test,
    predictions
))

In [ ]:
cm = confusion_matrix(y_test, predictions)

sns.heatmap(
    cm,
    annot=True,
    fmt='d'
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
importance = pd.DataFrame({
    'Feature': X_train.columns, # Use X_train.columns to match the features used in training
    'Importance': model.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance)

In [ ]:
probability = model.predict_proba(X_test)

risk_scores = pd.DataFrame({
    'EmployeeRiskScore': probability[:,1]
})

risk_scores.head()

In [ ]:
def recommendation(score):

    if score > 0.8:
        return "Immediate HR Intervention"

    elif score > 0.5:
        return "Retention Program"

    else:
        return "Low Risk"

risk_scores['Recommendation'] = (
    risk_scores['EmployeeRiskScore']
    .apply(recommendation)
)

risk_scores.head()

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    risk_scores['EmployeeRiskScore'],
    bins=20
)

plt.title("Employee Attrition Risk Distribution")
plt.show()

In [ ]:
prediction = model.predict(X_test)
probability = model.predict_proba(X_test)

In [ ]:
display(df.head())

In [ ]:
# To fix the syntax, you could use an underscore for the variable name, for example:
# success_score = (
#     Performance +
#     Training +
#     Attendance +
#     Satisfaction
# )

# However, 'Performance', 'Training', 'Attendance', and 'Satisfaction' are also not defined.
# You would need to ensure these variables hold numeric values or refer to columns in a DataFrame.

In [ ]:
!pip install openpyxl xlsxwriter

In [ ]:
!pip install reportlab

In [ ]:
!pip install shap

In [ ]:
import plotly.express as px

fig = px.bar(
    risk_scores,
    y="EmployeeRiskScore",
    title="Employee Retention Risk Dashboard"
)

fig.show()

In [ ]:
df['EmployeeID'] = df.index

# Normalize 'age' and 'fare' to a 0-1 scale
min_age = df['age'].min()
max_age = df['age'].max()
normalized_age = (df['age'] - min_age) / (max_age - min_age)

min_fare = df['fare'].min()
max_fare = df['fare'].max()
normalized_fare = (df['fare'] - min_fare) / (max_fare - min_fare)

# Calculate 'SuccessScore' as a weighted average, then scale to 0-100
# Assign higher weight to 'fare' as a proxy for 'success' in this context
df["SuccessScore"] = (
    normalized_fare * 0.7 +
    normalized_age * 0.3
) * 100

top_employees = df.sort_values(
    by="SuccessScore",
    ascending=False
)

print(top_employees[["EmployeeID", "SuccessScore"]].head(10))

In [ ]:
gender_count = df["sex"].value_counts()

print(gender_count)

gender_count.plot(
    kind="pie",
    autopct="%1.1f%%"
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.scatterplot(
    x="age",
    y="SuccessScore",
    data=df
)

plt.show()

In [ ]:
class_scores = df.groupby(
    "pclass"
)["SuccessScore"].mean()

print(
    class_scores.sort_values(
        ascending=False
    )
)

In [ ]:
future_attrition = (
    risk_scores["EmployeeRiskScore"] > 0.70
).sum()

print(
    "Expected Future Attrition:",
    future_attrition
)

In [ ]:
def interview_recommendation(score):

    if score > 85:
        return "Strong Hire"

    elif score > 70:
        return "Hire"

    elif score > 60:
        return "Consider"

    else:
        return "Reject"

# Create a sample candidate_df from the existing df (titanic data)
# using 'SuccessScore' as a proxy for 'InterviewScore'
candidate_df = df[['EmployeeID', 'SuccessScore']].copy()
candidate_df.rename(columns={'SuccessScore': 'InterviewScore'}, inplace=True)

candidate_df["Decision"] = (
    candidate_df["InterviewScore"]
    .apply(interview_recommendation)
)

print(candidate_df.head())

In [ ]:
df["ProductivityScore"] = df["SuccessScore"]

print(
    df[[
        "EmployeeID",
        "ProductivityScore"
    ]]
)

In [ ]:
df["EmployeeLifetimeValue"] = (
    df["fare"] *
    df["age"]
)

print(
    df[[
        "EmployeeID",
        "EmployeeLifetimeValue"
    ]]
)

In [ ]:
leave_analysis = df.groupby(
    "pclass"
)["age"].mean()

print(leave_analysis)

In [ ]:
def risk_category(score):

    if score > 0.80:
        return "Critical"

    elif score > 0.60:
        return "High"

    elif score > 0.40:
        return "Medium"

    return "Low"

risk_scores["RiskCategory"] = (
    risk_scores["EmployeeRiskScore"]
    .apply(risk_category)
)

In [ ]:
def hr_action(risk):

    if risk == "Critical":
        return "Immediate Meeting"

    elif risk == "High":
        return "Retention Bonus"

    elif risk == "Medium":
        return "Career Discussion"

    else:
        return "Monitor"

risk_scores["HRAction"] = (
    risk_scores["RiskCategory"]
    .apply(hr_action)
)

In [ ]:
with pd.ExcelWriter(
    "HR_AI_Report.xlsx"
) as writer:

    df.to_excel(
        writer,
        sheet_name="Employees",
        index=False
    )

    risk_scores.to_excel(
        writer,
        sheet_name="RiskAnalysis",
        index=False
    )

print("Report Generated")

In [ ]:
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer
)

from reportlab.lib.styles import (
    getSampleStyleSheet
)

pdf = SimpleDocTemplate(
    "Executive_Report.pdf"
)

styles = getSampleStyleSheet()

content = []

content.append(
    Paragraph(
        "AI Workforce Analytics Report",
        styles["Title"]
    )
)

content.append(
    Spacer(1,12)
)

content.append(
    Paragraph(
        "Generated using Machine Learning.",
        styles["Normal"]
    )
)

pdf.build(content)

print("PDF Created")

In [ ]:
!pip install shap

import shap

explainer = shap.TreeExplainer(model)

shap_values = explainer.shap_values(X_test)

shap.summary_plot(
    shap_values,
    X_test
)

In [ ]:
employee = pd.DataFrame({
    "pclass": [1], # Example: First class
    "sex": [0],    # Example: Female (assuming 0 for female after LabelEncoding)
    "age": [30.0], # Example: 30 years old
    "sibsp": [1],  # Example: 1 sibling/spouse aboard
    "parch": [1],  # Example: 1 parent/child aboard
    "fare": [50.0],# Example: Fare paid
    "embarked": [0], # Example: Embarked at Cherbourg (assuming 0 after LabelEncoding)
    "class": [0],  # Example: First class (assuming 0 after LabelEncoding)
    "who": [2],    # Example: Woman (assuming 2 for woman after LabelEncoding)
    "adult_male": [0], # Example: Not an adult male (as int)
    "embark_town": [0], # Example: Embark town Cherbourg (assuming 0 after LabelEncoding)
    "alive": [1],  # Example: Alive (assuming 1 for yes after LabelEncoding)
    "alone": [0]   # Example: Not alone (as int)
})

prediction = model.predict(employee)

print(prediction)

In [ ]:
high_risk = (
    risk_scores["RiskCategory"]
    == "Critical"
).sum()

print(
    f"{high_risk} employees require immediate intervention."
)

In [ ]:
corr = df.corr(
    numeric_only=True
)

print(corr["survived"].sort_values())

In [ ]:
print(
    "Employees:",
    len(df)
)

print(
    "Average Fare:",
    df["fare"].mean()
)

print(
    "Average Success Score:",
    df["SuccessScore"].mean()
)